In [2]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
monthly = pd.read_csv("/content/drive/MyDrive/field-shift/data/monthly_climate_full.csv", index_col=0,
                      parse_dates=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
avg_rain = monthly.groupby(monthly.index.month)["PRECTOTCORR"].mean()
avg_tmax = monthly.groupby(monthly.index.month)["T2M_MAX"].mean()
print(avg_rain.round(0))
print(avg_tmax.round(1))

1      28.0
2      27.0
3      38.0
4      22.0
5      19.0
6      78.0
7     192.0
8     153.0
9      95.0
10     18.0
11      9.0
12      8.0
Name: PRECTOTCORR, dtype: float64
1     24.2
2     28.9
3     36.6
4     42.4
5     46.9
6     47.4
7     44.1
8     40.2
9     38.3
10    36.6
11    31.3
12    26.9
Name: T2M_MAX, dtype: float64


In [6]:
crops = pd.DataFrame({
    "crop": ["wheat", "rice", "mung bean", "maize"],
    "months": [[10,11,12,1,2,3,4], [5,6,7,8,9,10], [7,8,9,10], [6,7,8,9,10]],
    "water_need_mm":  [310,1200,300,550],
    "heat_limit_c":   [32,37,40,35],
    "soil_points":    [2,1,5,2],
    "income_points":  [3,4,3,4],
    "source":         [
      "Wheat water: Hundal et al., PAU Ludhiana (312 mm ET). Heat limit: team estimate",
      "Rice water: Hussain et al. 2023 Sustainability 15(3):1768; PAU range 1250-1600 mm. Heat limit: team estimate",
      "Mung bean heat: Priya et al. 2023 Front. Plant Sci. Water need: team estimate",
      "Maize water: Iqbal et al. 2010 Agric. Water Mgmt 97(5):731-737 (range). Heat limit: team estimate"]
}).set_index("crop")

In [9]:
rows = []
for crop, c in crops.iterrows():
  rain_in_season = sum(avg_rain[m] for m in c["months"])
  irrigation_gap = max(0, c["water_need_mm"] - rain_in_season)
  hot_months = sum(avg_tmax[m] > c["heat_limit_c"] for m in c['months'])
  rows.append({"crop": crop, "season_rain_mm": round(rain_in_season),
               "irrigation_needed_mm": round(irrigation_gap), "heat_stress_months": hot_months})
crop_stats = pd.DataFrame(rows).set_index("crop")

In [15]:
rotations = {
    "A: Wheat-Rice (current)": ["wheat", "rice"],
    "B: Wheat-Mung bean": ["wheat", "mung bean"],
    "C: Wheat-Maize": ["wheat", "maize"]
}

def scale(s, higher_is_better=True):
  rng = s.max() - s.min()
  out = (s-s.min()) / rng * 100 if rng else s * 0 + 100
  return out if higher_is_better else 100 - out

table = pd.DataFrame({
    name: {
        "irrigation_mm": crop_stats.loc[cs, "irrigation_needed_mm"].sum(),
        "heat_months":   crop_stats.loc[cs, "heat_stress_months"].sum(),
        "soil":          crops.loc[cs, "soil_points"].mean(),
        "income":        crops.loc[cs, "income_points"].mean(),
    } for name, cs in rotations.items()
}).T

scores = pd.DataFrame({
    "Water":  scale(table["irrigation_mm"], higher_is_better=False),
    "Risk":   scale(table["heat_months"], higher_is_better=False),
    "Soil":   scale(table["soil"]),
    "Income": scale(table["income"])
})

w = {"Water": 0.3, "Soil": 0.3, "Income": 0.2, "Risk": 0.2}
scores["Overall"] = sum(scores[k] * v for k, v in w.items())
scores.round()

,Water,Risk,Soil,Income,Overall
A: Wheat-Rice (current),0.0,0.0,0.0,100.0,20.0
B: Wheat-Mung bean,100.0,100.0,100.0,0.0,80.0
C: Wheat-Maize,98.0,0.0,25.0,100.0,57.0


In [16]:
scores.to_csv("/content/drive/MyDrive/field-shift/data/rotation_scores.csv")
table.to_csv("/content/drive/MyDrive/field-shift/data/rotation_evidence.csv")
crops.to_csv("/content/drive/MyDrive/field-shift/data/crops.csv")
crop_stats.to_csv("/content/drive/MyDrive/field-shift/data/crop_stats.csv")

In [22]:
def run_scenario(rain_factor=1.0, temp_add=0.0, weights=w):
  rows = []
  for crop, c in crops.iterrows():
    rain = sum(avg_rain[m] * rain_factor for m in c["months"])
    gap = max(0, c["water_need_mm"] - rain)
    hot = sum((avg_tmax[m] + temp_add) > c["heat_limit_c"] for m in c["months"])
    rows.append({"crop": crop,
                 "irrigation_needed_mm": gap,
                 "heat_stress_months": hot
                 })
  cs = pd.DataFrame(rows).set_index("crop")

  t = pd.DataFrame({
      name: {
          "irrigation_mm": cs.loc[crop_list, "irrigation_needed_mm"].sum(),
          "heat_months":   cs.loc[crop_list, "heat_stress_months"].sum(),
          "soil":          crops.loc[crop_list, "soil_points"].mean(),
          "income":        crops.loc[crop_list, "income_points"].mean()
      } for name, crop_list in rotations.items()
  }).T

  s = pd.DataFrame({
      "Water":  scale(t["irrigation_mm"], higher_is_better=False),
      "Risk":   scale(t["heat_months"], higher_is_better=False),
      "Soil":   scale(t["soil"]),
      "Income": scale(t["income"])
  })
  s["Overall"] = sum(s[k] * v for k, v in weights.items())
  return s.round(0)
print("Today:")
print(run_scenario())
print("20% less rain:")
print(run_scenario(rain_factor=0.8))
print("+2°C hotter:")
print(run_scenario(temp_add=2))

Today:
                         Water   Risk   Soil  Income  Overall
A: Wheat-Rice (current)    0.0    0.0    0.0   100.0     20.0
B: Wheat-Mung bean       100.0  100.0  100.0     0.0     80.0
C: Wheat-Maize            98.0    0.0   25.0   100.0     57.0
20% less rain:
                         Water   Risk   Soil  Income  Overall
A: Wheat-Rice (current)    0.0    0.0    0.0   100.0     20.0
B: Wheat-Mung bean       100.0  100.0  100.0     0.0     80.0
C: Wheat-Maize            84.0    0.0   25.0   100.0     53.0
+2°C hotter:
                         Water   Risk   Soil  Income  Overall
A: Wheat-Rice (current)    0.0    0.0    0.0   100.0     20.0
B: Wheat-Mung bean       100.0  100.0  100.0     0.0     80.0
C: Wheat-Maize            98.0   33.0   25.0   100.0     63.0
